### RapidFire AI Tutorial Use Case: SFT for Customer Support Q&A Chatbot

##### Note: Minimum hardware required for this notebook is 2*A10 GPUs

In [1]:
from rapidfireai import Experiment
from rapidfireai.automl import List, RFGridSearch, RFModelConfig, RFLoraConfig, RFSFTConfig

### Load Dataset and Specify Train and Eval Partitions

In [2]:
from datasets import load_dataset

dataset=load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")

# Select a subset of the dataset for demo purposes
train_dataset=dataset["train"].select(range(128))
eval_dataset=dataset["train"].select(range(128,144))
train_dataset=train_dataset.shuffle(seed=42)
eval_dataset=eval_dataset.shuffle(seed=42)

### Define Data Processing Function

In [3]:
def sample_formatting_function(row):
    """Function to preprocess each example from dataset"""
    # Special tokens for formatting
    SYSTEM_PROMPT = "You are a helpful and friendly customer support assistant. Please answer the user's query to the best of your ability."
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row["instruction"]},
            
        ],
        "completion": [
            {"role": "assistant", "content": row["response"]}
        ]
    }

### Initialize Experiment

In [4]:
# Every experiment instance must be uniquely named
experiment = Experiment(experiment_name="exp1-chatqa-fsdp-lite")

Experiment exp1-chatqa-fsdp-lite is currently running. Returning the same experiment object.


### Define Custom Eval Metrics Function

In [5]:
def sample_compute_metrics(eval_preds):  
    """Optional function to compute eval metrics based on predictions and labels"""
    predictions, labels = eval_preds

    # Standard text-based eval metrics: Rouge and BLEU
    import evaluate
    rouge = evaluate.load("rouge")
    bleu = evaluate.load("bleu")

    rouge_output = rouge.compute(predictions=predictions, references=labels, use_stemmer=True)
    rouge_l = rouge_output["rougeL"]
    bleu_output = bleu.compute(predictions=predictions, references=labels)
    bleu_score = bleu_output["bleu"]

    return {
        "rougeL": round(rouge_l, 4),
        "bleu": round(bleu_score, 4),
    }

### Define Multi-Config Knobs for Model, LoRA, and SFT Trainer using RapidFire AI Wrapper APIs

In [6]:
# 2 LoRA PEFT configs lite with different adapter capacities
peft_configs_lite = List([
    RFLoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=["q_proj", "v_proj"],  
        bias="none"
    ),
    RFLoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],  
        bias="none"
    ),
])

# 2 learning rates  and 2 peft configs = 4 combinations in total
config_set_lite = List([
    RFModelConfig(
        model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",  # 1.1B model
        peft_config=peft_configs_lite,
        training_args=RFSFTConfig(
            learning_rate=List([1e-3, 1e-4]),  
            lr_scheduler_type="linear",
            per_device_train_batch_size=2,
            per_device_eval_batch_size=2,
            num_train_epochs=1,
            gradient_accumulation_steps=1,   
            logging_steps=1,
            eval_strategy="steps",
            eval_steps=8,
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
            bf16=True,
            fp16=False,
            fsdp="full_shard auto_wrap",
            fsdp_config={"backward_prefetch": "backward_pre","forward_prefetch": True,"use_orig_params": False,  "cpu_ram_efficient_loading": True,"offload_params": True, "sync_module_states": True,"limit_all_gathers": True, "sharding_strategy": "FULL_SHARD",
                "auto_wrap_policy": "TRANSFORMER_BASED_WRAP"},   
        ),
        model_type="causal_lm",
        model_kwargs={"device_map":None, "torch_dtype": "bfloat16", "use_cache": False},
        formatting_func=sample_formatting_function,
        compute_metrics=sample_compute_metrics,
        generation_config = { # This is for text based evaluation/prediction for causal_lm models
            "max_new_tokens": 256,
            "do_sample": False,
            "use_cache": False,
        },
        num_gpus=2,
    )
])


#### Define Model Creation Function for All Model Types Across Configs

In [7]:

def sample_create_model(model_config): 
     """Function to create model object for any given config; must return tuple of (model, tokenizer)"""
     from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForMaskedLM, GptOssForCausalLM

     model_name = model_config["model_name"]
     model_type = model_config["model_type"]
     model_kwargs = model_config["model_kwargs"]
     
 
     if model_type == "causal_lm":
          model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
     elif model_type == "seq2seq_lm":
          model = AutoModelForSeq2SeqLM.from_pretrained(model_name, **model_kwargs)
     elif model_type == "masked_lm":
          model = AutoModelForMaskedLM.from_pretrained(model_name, **model_kwargs)
     elif model_type == "custom":
          # Handle custom model loading logic, e.g., loading your own checkpoints
          # model = ...
          pass
     else:
          # Default to causal LM
          model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
      
     tokenizer = AutoTokenizer.from_pretrained(model_name)
     return (model,tokenizer)

#### Generate Config Group

In [8]:
# Simple grid search across all sets of config knob values = 4 combinations in total
config_group = RFGridSearch(
    configs=config_set_lite,
    trainer_type="SFT"
)

### Run Multi-Config Training

In [9]:
# Launch training of all configs in the config_group with swap granularity of 4 chunks
experiment.run_fit(config_group, sample_create_model, train_dataset, eval_dataset, num_chunks=4, seed=42)

Started 1 worker processes successfully
Created workers


ExperimentException: Error running fit: Run 13 requires 2 workers but only 1 workers are available, traceback: Traceback (most recent call last):
  File "/home/rapid-fire/rapidfireai/rapidfireai/experiment.py", line 447, in run_fit
    controller.run_fit(param_config, create_model_fn, train_dataset, eval_dataset, num_chunks, seed, num_gpus, monte_carlo_simulations)
  File "/home/rapid-fire/rapidfireai/rapidfireai/fit/backend/controller.py", line 654, in run_fit
    scheduler = Scheduler(
                ^^^^^^^^^^
  File "/home/rapid-fire/rapidfireai/rapidfireai/fit/backend/scheduler.py", line 64, in __init__
    raise ValueError(
ValueError: Run 13 requires 2 workers but only 1 workers are available


### End Current Experiment

In [ ]:
experiment.end()    